## Task 1: Explore the Data

In [ ]:
# Import libraries and load the Superstore dataset from a CSV file
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load the dataset (note: you will download from Kaggle and upload to Colab)
df = pd.read_csv('Sample - Superstore.csv', encoding='latin1')

In [ ]:
# Inspect dataset: shape, data types, and basic statistics
print('Dataset shape:', df.shape)
print('\nData types:')
print(df.dtypes)
print('\nBasic statistics:')
print(df.describe())

In [ ]:
# Select numerical features as predictors for Sales
features = ['Quantity', 'Discount', 'Profit']
X_explore = df[features]
y_explore = df['Sales']
print('Selected features:', features)
print('Target variable: Sales')

In [ ]:
# Check for missing values in features and target
print('Missing values:')
print(df[features + ['Sales']].isnull().sum())

In [ ]:
# Create and visualize correlation matrix between predictors and Sales
corr = df[['Sales', 'Quantity', 'Discount', 'Profit']].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0)
plt.title('Feature Correlations with Sales')
plt.tight_layout()
plt.show()
print('\nCorrelation matrix:')
print(corr)

## Task 2: Build a Linear Regression Model

In [ ]:
# Define features X and target y for the model
X = df[['Quantity', 'Discount', 'Profit']]
y = df['Sales']
print('Features shape:', X.shape)
print('Target shape:', y.shape)

In [ ]:
# Split data into training and testing sets (80/20 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Training set size: {X_train.shape[0]}')
print(f'Testing set size: {X_test.shape[0]}')

In [ ]:
# Fit linear regression model on training data and make predictions on test data
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print('Linear model trained successfully.')

In [ ]:
# Compute regression metrics to evaluate model performance
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"MSE: {mse:.2f}, RMSE: {rmse:.2f}, MAE: {mae:.2f}, R2: {r2:.4f}")

## Task 3: Interpret the Coefficients

In [ ]:
# Print model coefficients and intercept for interpretation
print('Model Coefficients:')
for feature, coef in zip(X.columns, model.coef_):
    print(f"  {feature}: {coef:.4f}")
print(f"\nIntercept: {model.intercept_:.4f}")

In [ ]:
# Interpret coefficients in business terms
print("\n=== Coefficient Interpretation ===")
print(f"\n1. Quantity (coef: {model.coef_[0]:.4f}):")
print(f"   A one-unit increase in Quantity is associated with a ${model.coef_[0]:.2f} increase in Sales,")
print("   holding Discount and Profit constant. This positive relationship makes intuitive sense:")
print("   selling more items directly increases total sales.")
print(f"\n2. Discount (coef: {model.coef_[1]:.4f}):")
print(f"   A one-unit increase in Discount is associated with a ${model.coef_[1]:.2f} change in Sales,")
print("   holding other features constant. A negative coefficient suggests discounting may reduce")
print("   overall sales value, possibly because discounts reduce the revenue per unit sold.")
print(f"\n3. Profit (coef: {model.coef_[2]:.4f}):")
print(f"   A one-unit increase in Profit is associated with a ${model.coef_[2]:.2f} increase in Sales,")
print("   holding other features constant. The strong positive relationship suggests profitability")
print("   is tightly linked to sales volume.")

## Task 4: Try Polynomial Regression

In [ ]:
# Create polynomial features of degree 2 to capture non-linear relationships
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)
print(f'Original features: {X_train.shape[1]}')
print(f'Polynomial features (degree 2): {X_train_poly.shape[1]}')

In [ ]:
# Train polynomial regression model on the expanded feature space
model_poly = LinearRegression()
model_poly.fit(X_train_poly, y_train)
y_pred_poly = model_poly.predict(X_test_poly)
print('Polynomial model trained successfully.')

In [ ]:
# Compute metrics for polynomial model and compare with linear
mse_poly = mean_squared_error(y_test, y_pred_poly)
rmse_poly = np.sqrt(mse_poly)
mae_poly = mean_absolute_error(y_test, y_pred_poly)
r2_poly = r2_score(y_test, y_pred_poly)

# Create comparison table
comparison = pd.DataFrame({
    'Metric': ['MSE', 'RMSE', 'MAE', 'R2'],
    'Linear Regression': [f"{mse:.2f}", f"{rmse:.2f}", f"{mae:.2f}", f"{r2:.4f}"],
    'Polynomial Regression': [f"{mse_poly:.2f}", f"{rmse_poly:.2f}", f"{mae_poly:.2f}", f"{r2_poly:.4f}"]
})
print('\n=== Model Comparison ==="')
print(comparison.to_string(index=False))

In [ ]:
# Analysis of polynomial regression results
print("\n=== Polynomial vs Linear Analysis ===")
if r2_poly > r2:
    print(f"The polynomial model shows improvement: R² increased from {r2:.4f} to {r2_poly:.4f}.")
    print("This suggests that non-linear relationships between features and Sales exist.")
    print("The polynomial features capture interaction effects and squared terms that better explain sales variation.")
else:
    print(f"The linear model performs as well or better: R² for linear = {r2:.4f}, polynomial = {r2_poly:.4f}.")
    print("This indicates that the relationships are primarily linear, and adding complexity")
    print("does not meaningfully improve predictions. Simpler models are often preferable (Occam's Razor).")

## Task 5: Visualize

In [ ]:
# Predicted vs Actual scatter plot for linear regression
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.5, s=20, label='Predictions')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Sales')
plt.ylabel('Predicted Sales')
plt.title('Predicted vs Actual Sales (Linear Regression)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Residual plot to diagnose model quality
residuals = y_test - y_pred
plt.figure(figsize=(8, 6))
plt.scatter(y_pred, residuals, alpha=0.5, s=20)
plt.axhline(y=0, color='r', linestyle='--', lw=2)
plt.xlabel('Predicted Sales')
plt.ylabel('Residuals')
plt.title('Residual Plot (Linear Regression)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Interpretation of diagnostic plots
print("\n=== Diagnostic Interpretation ===")
print("\nPredicted vs Actual Plot:")
print("- Points clustered along the diagonal (red dashed line) indicate good predictions.")
print("- Points scattered far from the diagonal suggest the model struggles with certain ranges.")
print("- The spread around the diagonal shows prediction uncertainty.")
print("\nResidual Plot:")
print("- Random scatter around y=0 (red line) indicates the model is working well.")
print("- A U-shaped or cone-shaped pattern suggests non-linearity or heteroscedasticity.")
print("- Systematic patterns indicate the model is missing important features or relationships.")
print(f"\nResidual Statistics:")
print(f"  Mean of residuals: {residuals.mean():.4f} (should be close to 0)")
print(f"  Std of residuals: {residuals.std():.4f}")
print(f"  Min residual: {residuals.min():.2f}")
print(f"  Max residual: {residuals.max():.2f}")